## Evaluate pytorch model from Fourier series


In [ ]:
# Importing necessary packages
import sys
import os
import importlib
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import r2_score,mean_squared_error
import matplotlib.pyplot as plt


path_base = Path(os.getcwd()) 
# Current path for importing custom functions
sys.path.insert(0, str(path_base / "clc_functions")) 

import input_transform
importlib.reload(input_transform)
from input_transform import inputs_transform #, inverse_transform_clc 
#Remark: inverse_transform is hardcoded in pytorch model, i.e. does not need to be importet


In [ ]:

# Folder in which to find the test inputs
test_inputs_folder = 'test_data/'


transform_output = True
name_out_transf = ''
if transform_output:
    name_out_transf = '_transformedCLC'

### Interval within which transformed clc should be bounded
bound_output = [0.0, 1.0]

### Upper bound for input transformation
transform_input = True
upperbound = np.pi
upperbound_name = '1p0pi'

batch_size = 100
n_batch_name = str(batch_size)

# Learning rate
learning_rate = 0.001
learning_rate_name = '0p001'

### Kept features
features_kept = ['hus', 'clw', 'cli', 'ta', 'pa', 'hwind']
no_of_features = len(features_kept)

### Architecture specifications
no_qubits = no_of_features

ind_features = [0,1,2,3,4,6]

In [ ]:

# Load test data

namefilein = 'cirrus_inputs_raw_8features.npy'
namefileout = 'cirrus_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_cirrus_full = np.load(path_file)
test_inputs_cirrus = test_inputs_cirrus_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_cirrus = np.load(path_file)
no_testing_data_cirrus = test_inputs_cirrus.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_cirrus[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_cirrus[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_cirrus = np.squeeze(np.logical_not(IIIlowclt))
indsIII_cirrus = np.where(III_cirrus)[0]
no_test_samples_to_evaluate_cirrus = np.sum(III_cirrus)
print('No. test samples to evaluate (cirrus): ', no_test_samples_to_evaluate_cirrus)

namefilein = 'cumulus_inputs_raw_8features.npy'
namefileout = 'cumulus_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_cumulus_full = np.load(path_file)
test_inputs_cumulus = test_inputs_cumulus_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_cumulus = np.load(path_file)
no_testing_data_cumulus = test_inputs_cumulus.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_cumulus[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_cumulus[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_cumulus = np.squeeze(np.logical_not(IIIlowclt))
indsIII_cumulus = np.where(III_cumulus)[0]
no_test_samples_to_evaluate_cumulus = np.sum(III_cumulus)
print('No. test samples to evaluate (cumulus): ', no_test_samples_to_evaluate_cumulus)

namefilein = 'deepconv_inputs_raw_8features.npy'
namefileout = 'deepconv_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_deepconv_full = np.load(path_file)
test_inputs_deepconv = test_inputs_deepconv_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_deepconv = np.load(path_file)
no_testing_data_deepconv = test_inputs_deepconv.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_deepconv[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_deepconv[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_deepconv = np.squeeze(np.logical_not(IIIlowclt))
indsIII_deepconv = np.where(III_deepconv)[0]
no_test_samples_to_evaluate_deepconv = np.sum(III_deepconv)
print('No. test samples to evaluate (deepconv): ', no_test_samples_to_evaluate_deepconv)

namefilein = 'stratus_inputs_raw_8features.npy'
namefileout = 'stratus_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_stratus_full = np.load(path_file)
test_inputs_stratus = test_inputs_stratus_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_stratus = np.load(path_file)
no_testing_data_stratus = test_inputs_stratus.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_stratus[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_stratus[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_stratus = np.squeeze(np.logical_not(IIIlowclt))
indsIII_stratus = np.where(III_stratus)[0]
no_test_samples_to_evaluate_stratus = np.sum(III_stratus)
print('No. test samples to evaluate (stratus): ', no_test_samples_to_evaluate_stratus)

### Transform inputs if needed
if transform_input:
    bound_input = [0.0, upperbound]
    bounds = [bound_input for _ in features_kept]
    test_inputs_cirrus_t = inputs_transform(test_inputs_cirrus, features_kept, bounds)
    test_inputs_cumulus_t = inputs_transform(test_inputs_cumulus, features_kept, bounds)
    test_inputs_deepconv_t = inputs_transform(test_inputs_deepconv, features_kept, bounds)
    test_inputs_stratus_t = inputs_transform(test_inputs_stratus, features_kept, bounds)

### Convert testing data to jax numpy arrays
jnp_test_inputs_cirrus = np.asarray(test_inputs_cirrus_t)
jnp_test_inputs_cumulus = np.asarray(test_inputs_cumulus_t)
jnp_test_inputs_deepconv = np.asarray(test_inputs_deepconv_t)
jnp_test_inputs_stratus = np.asarray(test_inputs_stratus_t)



In [ ]:

### PQC architecture layout
# No of shots for circuit evaluation
no_shots = 1000 # or #inf
#No of shots that were used in training 
no_shots_training = 'inf'
#Architecture to be used 'ZZXY' or 'XYZ'
name_arch = 'XYZ'

### PQC architecture layout + optimal params
if name_arch == 'XYZ':
    #pqc_layout = pqcs.XYZ_circuit; 
    name_arch = 'XYZ';  n_enc = 4;  n_dec = 2; 
    if no_shots_training == 'inf':
        best_exp = 1 
    else:
        best_exp = 4
elif name_arch == 'ZZXY':
    #pqc_layout = pqcs.ZZXY_circuit;  
    name_arch = 'ZZXY';  n_enc = 2;  n_dec = 5;
    if no_shots_training == 'inf':
        best_exp = 6 
    else:
        best_exp = 3

In [ ]:
filename = 'qfm_' + str(no_qubits) +'f_'+ name_arch + '_'+ str(no_shots_training)+'t_'+ str(no_shots) +'s_test'+str(best_exp)

data = np.load(filename + '.npy',allow_pickle = True)


In [ ]:
qfm = data.item()
O = qfm['spectrum']
Ccos = qfm['Cos']
Csin = qfm['Sin']
A = qfm['weights']
b = qfm['bias']
f_Omega = qfm['trunc_frequencies']
Omega = torch.from_numpy(O)

Cos = torch.from_numpy(Ccos)
Sin = torch.from_numpy(Csin)


In [ ]:
   
class FourierLinearModel(nn.Module):
    def __init__(self, O, C_cos, C_sin, weights, bias):
        """
        O: (K,d) spectrum,
        C_cos:(d,K) cosine coefficients
        C_sin:(d,K) sine coefficients
        A: (1,d)weights
        b: (1,)  bias
        """
        super().__init__()

        self.register_buffer("O", torch.as_tensor(O, dtype=torch.float32))
        self.register_buffer("C_cos", torch.as_tensor(C_cos, dtype=torch.float32))
        self.register_buffer("C_sin", torch.as_tensor(C_sin, dtype=torch.float32))
        self.register_buffer("weights", torch.as_tensor(weights, dtype=torch.float32))
        self.register_buffer("bias", torch.as_tensor(bias, dtype=torch.float32))

    def forward(self, x):
        """
        x : (batch_size, d)
        returns: (batch_size, 1)
        """
       
        x = torch.as_tensor(x,dtype = torch.float32)
        phase = x @ self.O.T

        # Fourier features
        cos_part = torch.cos(phase)
        sin_part = torch.sin(phase)

        # Fourier model output (batch_size, d)
        F = cos_part @ self.C_cos.T + sin_part @ self.C_sin.T

        # Linear layer A F(x) + b
        y = F @ self.weights + self.bias

        #Post processing
        y = torch.max(y, torch.zeros_like(y))
        y = torch.min(y, torch.ones_like(y))

        #inverse transform clc

        a = 1.29407913
        b = -3.20011015
        c = 0.70308237
        vx = (0.5 * torch.sin(np.pi*(y - 0.5)) + 0.5)**(1.0/c)
        gx = (np.exp(b) - 1.0) * vx
        xp = torch.log(gx + 1.0) / b
        x = xp**(1.0/a)
        return x

# Test model and compare MSE

In [ ]:
mse = np.ones((4,len(f_Omega)))
r2 = np.ones((4,len(f_Omega)))
for index,i in enumerate(f_Omega):
    f = i//2
    Omega_T = torch.from_numpy(O[0:f,:])
    Cos_T = torch.from_numpy(Ccos[0:f,:])
    Sin_T = torch.from_numpy(Csin[0:f,:])
    model = FourierLinearModel(Omega_T,Cos_T.T,Sin_T.T,A,b) 
    batch_size = 100000
    
    
    pred_test_outputs = np.zeros(no_testing_data_cirrus) 
    no_batches_test = int(np.floor(no_test_samples_to_evaluate_cirrus / batch_size))!
    for kk in range(0, no_batches_test-1):
        III_to_eval = indsIII_cirrus[kk*batch_size:(kk+1)*batch_size] !
        ins_batch = jnp_test_inputs_cirrus[III_to_eval, :] 

        outs_batch =  model(ins_batch)  
    
        pred_test_outputs[III_to_eval] = outs_batch
     
    III_to_eval = indsIII_cirrus[(no_batches_test-1)*batch_size:] 
    ins_batch = jnp_test_inputs_cirrus[III_to_eval, :]
    
    
    
    outs_batch = model(ins_batch)
    pred_test_outputs[III_to_eval] = outs_batch
    mse_c = mean_squared_error(test_outputs_cirrus[indsIII_cirrus],pred_test_outputs[indsIII_cirrus])
    r2_c=r2_score(test_outputs_cirrus[indsIII_cirrus],pred_test_outputs[indsIII_cirrus])
    mse[0,index] = mse_c
    r2[0,index] = r2_c
print('done')
    


In [ ]:
fig, axs = plt.subplots(1,1)
axs.plot(f_Omega, mse[0,:])
plt.show()


In [ ]:
for index,i in enumerate(f_Omega):
    f = i//2
    Omega_T = torch.from_numpy(O[0:f,:])
    Cos_T = torch.from_numpy(Ccos[0:f,:])
    Sin_T = torch.from_numpy(Csin[0:f,:])
    model = FourierLinearModel(Omega_T,Cos_T.T,Sin_T.T,A,b) 
    batch_size = 100000
    
    
    pred_test_outputs = np.zeros(no_testing_data_deepconv) 
    no_batches_test = int(np.floor(no_test_samples_to_evaluate_deepconv / batch_size)) 
    for kk in range(0, no_batches_test-1):
        III_to_eval = indsIII_deepconv[kk*batch_size:(kk+1)*batch_size] 
        ins_batch = jnp_test_inputs_deepconv[III_to_eval, :]
        
    
        outs_batch =  model(ins_batch)
    
    
        pred_test_outputs[III_to_eval] = outs_batch
     
    III_to_eval = indsIII_deepconv[(no_batches_test-1)*batch_size:] 
    ins_batch = jnp_test_inputs_deepconv[III_to_eval, :]
    
    
    
    outs_batch = model(ins_batch)
    pred_test_outputs[III_to_eval] = outs_batch
    mse_c = mean_squared_error(test_outputs_deepconv[indsIII_deepconv],pred_test_outputs[indsIII_deepconv])
    r2_c=r2_score(test_outputs_deepconv[indsIII_deepconv],pred_test_outputs[indsIII_deepconv])
    mse[1,index] = mse_c
    r2[1,index] = r2_c
print('done')
    

In [ ]:
for index,i in enumerate(f_Omega):
    f = i//2
    Omega_T = torch.from_numpy(O[0:f,:])
    Cos_T = torch.from_numpy(Ccos[0:f,:])
    Sin_T = torch.from_numpy(Csin[0:f,:])
    model = FourierLinearModel(Omega_T,Cos_T.T,Sin_T.T,A,b) 
    batch_size = 100000
    
    
    pred_test_outputs = np.zeros(no_testing_data_stratus) 
    no_batches_test = int(np.floor(no_test_samples_to_evaluate_stratus / batch_size))
    for kk in range(0, no_batches_test-1):
        III_to_eval = indsIII_stratus[kk*batch_size:(kk+1)*batch_size]
        ins_batch = jnp_test_inputs_stratus[III_to_eval, :]      
    
        outs_batch =  model(ins_batch)
    
        pred_test_outputs[III_to_eval] = outs_batch
     
    III_to_eval = indsIII_stratus[(no_batches_test-1)*batch_size:] 
    ins_batch = jnp_test_inputs_stratus[III_to_eval, :]
    
    
    
    outs_batch = model(ins_batch)
    pred_test_outputs[III_to_eval] = outs_batch
    mse_c = mean_squared_error(test_outputs_stratus[indsIII_stratus],pred_test_outputs[indsIII_stratus])
    r2_c=r2_score(test_outputs_stratus[indsIII_stratus],pred_test_outputs[indsIII_stratus])
    mse[2,index] = mse_c
    r2[2,index] = r2_c
print('done')
    

In [ ]:
for index,i in enumerate(f_Omega):
    f = i//2
    Omega_T = torch.from_numpy(O[0:f,:])
    Cos_T = torch.from_numpy(Ccos[0:f,:])
    Sin_T = torch.from_numpy(Csin[0:f,:])
    model = FourierLinearModel(Omega_T,Cos_T.T,Sin_T.T,A,b) 
    batch_size = 100000
    
    
    pred_test_outputs = np.zeros(no_testing_data_cumulus) 
    no_batches_test = int(np.floor(no_test_samples_to_evaluate_cumulus/ batch_size)) 
    for kk in range(0, no_batches_test-1):
        III_to_eval = indsIII_cumulus[kk*batch_size:(kk+1)*batch_size]
        ins_batch = jnp_test_inputs_cumulus[III_to_eval, :] 
        
    
        outs_batch =  model(ins_batch)
    
    
        pred_test_outputs[III_to_eval] = outs_batch
     
    III_to_eval = indsIII_cumulus[(no_batches_test-1)*batch_size:] 
    ins_batch = jnp_test_inputs_cumulus[III_to_eval, :]
    
    
    
    outs_batch = model(ins_batch)
    pred_test_outputs[III_to_eval] = outs_batch
    mse_c = mean_squared_error(test_outputs_cumulus[indsIII_cumulus],pred_test_outputs[indsIII_cumulus])
    r2_c=r2_score(test_outputs_cumulus[indsIII_cumulus],pred_test_outputs[indsIII_cumulus])
    mse[3,index] = mse_c
    r2[3,index] = r2_c
print('done')
    

In [ ]:
fig, axs = plt.subplots(1,1)
axs.plot(f_Omega, mse[0,:])
axs.plot(f_Omega, mse[1,:])
axs.plot(f_Omega, mse[2,:])
axs.plot(f_Omega, mse[3,:])
plt.show()

np.argmin(mse,axis = 1)